In [ ]:
import os
import sys
import re
from abc import abstractmethod
from dataclasses import dataclass
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
import spacy
from collections import Counter
from itertools import chain
from abc import ABC
from collections.abc import Iterable

# sys.path.insert(0, os.path.abspath('../code'))
from data_sets_loaders import CsvLoader, TextDataBootstrapper, TextDatasetHelperMixin

### variables

In [ ]:
data_url = "../data/valuations.csv"

In [ ]:
valuations = pd.read_csv(data_url)
valuations

In [ ]:
loader = CsvLoader(data_url=data_url, x_cols=["ticker", "company", "news"], y_cols=["y"])
x, y = loader.load()
x

In [ ]:
t = TextDatasetHelperMixin().tokenize_texts(x)

In [ ]:
v = TextDatasetHelperMixin().build_vocabulary(t, 30000)

In [ ]:
t

In [ ]:
bootstrapper = TextDataBootstrapper(loader=loader, validation_split=0.2, test_split=0.1)

In [ ]:
# train_loader, validation_loader, test_loader = bootstrapper.get_dataloaders()

In [ ]:
bootstrapper.train_ds[0]

In [ ]:
c = np.array([2])

In [ ]:
data = np.array(
    [
        [["A", "B", "C"]],
        [["Apple", "Bolton", "Coreweave"]],
        [
            [
                "MicroStrategy Wants to Massively Dilute Shareholders",
                "Arm Announces Appointment of Eric Hayes as Executive",
                "Vice President, Operations: Arm Holdings plc today",
            ]
        ],
    ]
)

In [ ]:
class TextHelper:
    def tokenize_texts(self, texts: np.ndarray, spacy_model: str = "en_core_web_sm") -> list[list[str]]:
        """Tokenize texts using spaCy, handling 1D and 2D arrays."""
        result = []
        if type(texts) is np.ndarray and texts.ndim > 1:
            for i in range(texts.shape[1]):
                result.append(self.tokenize_texts(texts[:, i], spacy_model=spacy_model))
        elif type(texts) is np.ndarray and texts.ndim == 1:
            result.append(self._tokenize(texts, spacy_model=spacy_model))
        else:
            raise ValueError("Input texts must be a numpy array.")
        return result

    def _tokenize(self, texts: np.ndarray, spacy_model: str) -> list[list[str]]:
        """Tokenize texts using spaCy.

        Args:
            texts: Iterable of text strings to tokenize.
            spacy_model: Name of the spaCy model to use (default: "en_core_web_sm").

        Returns:
            List of tokenized texts, where each text is a list of token strings.
        """
        nlp = spacy.load(spacy_model)
        tokenized_texts = []

        # Convert to list of Python strings, handling NaN values
        text_list = []
        for text in texts:
            if pd.isna(text) or text is np.nan:
                text_list.append("")
            else:
                text_list.append(str(text))

        for doc in nlp.pipe(text_list, disable=["parser", "ner"]):
            tokens = []
            if doc is not np.nan:
                tokens = [token.text for token in doc]
            tokenized_texts.append(tokens)
        return tokenized_texts

In [ ]:
data

In [ ]:
TextHelper().tokenize_texts(data)